# mem pipeline：各环节的想法、要求和缺口

**工作文档** · 建立于 2026-11-11 · 配套：[演示 notebook](../../report/2026-11-10_mem_pipeline_demo.ipynb)（看图和数字）、[讨论稿](../../docs/mem/mem分支_问题定义与讨论稿_2026-11-10.md)第 17 节（环节清单）

这份 notebook 只有文字，不需要运行。它把整条流程拆成 15 个环节，每个环节写清楚四件事：**想法**、**要求**、**现在怎么做**、**缺什么**，另附待定问题和建议先做的试验。

**每个环节的示例图**在 [pipeline_examples.ipynb](pipeline_examples.ipynb) 里，编号与这里相同。意见仍然写在这份文档里。

**怎么用**

- 每个环节后面有一个 **✏️ 你的更新** 格子，想法、修改意见、决定都可以写在那里；正文写得不对的地方也可以直接改。
- 改完保存，告诉我"读一下 pipeline"。我会对照上一次提交找出你改了什么，逐条回应，再把结论写回对应环节，并同步到 GitHub 和服务器。
- 我更新这个文件之前会先同步你的改动，不会覆盖你写的内容。

**状态标记**

| 标记 | 含义 |
| --- | --- |
| 真实 | 只用输入（视频、简单深度、设备位姿）算出 |
| ORACLE | 目前直接取自 SceneFun3D 的标注，以后要换成模型预测 |
| 部分 | 有一部分，核心内容还没有 |
| 缺 | 目前没有这个环节 |

"要求"都是我根据前两轮讨论起草的草案，不是已定的结论。数字全部来自场景 421069 的一个视频（42444490），只说明误差从哪里来，不是结果。

**更新记录**

- **2026-11-11：** 总览后面新增"总览的调整"一节（需求理解、总览 v2 提案、相关研究、我的观点和待定点）；文末新增环节 14（交互检测）和环节 15（事件与状态时间线）。环节 1–13 的正文没有改。

- **2026-11-11（第三轮讨论）：** 记录你的决定：需求理解正确、接受"不做稠密重建"的说法、精度分两个阶段、阶段选方案 B；存储倾向 implicit（隐式）的形式，具体形式未定。总览后面新增"记忆的存储形式"一节：implicit 的几种含义、参考文章的简介、隐式与显式的对照证据、放到我们流程里的做法和待定点。

- **2026-11-11（第四轮讨论）：** 存储形式定为 L2（每个关键帧的图，加部件 token 和薄索引）；几何量读取时现算；检索先用 affordance token 和传统语义找关键帧，再交给视觉语言模型；两种特征都存。据此改写了总览和全部 15 个环节。每个环节在场景 421069 上有了一个示例，见 [pipeline_examples.ipynb](pipeline_examples.ipynb)（编号相同，带图）。
- **2026-11-12（第五轮讨论）：** 你确认总览采用 L2、环节 14 和 15 保留；重建对照和关键帧的修改讨论之后再定。研究层面的点评、理想框架和路线写在[讨论稿](../../docs/mem/mem分支_问题定义与讨论稿_2026-11-10.md)第 20 节，是提案。
- **2026-11-12（第六轮讨论）：** 采用"功能记忆"的主张，批准阶段 1。实验规格在[讨论稿](../../docs/mem/mem分支_问题定义与讨论稿_2026-11-10.md)第 21 节。
- **2026-11-12（阶段 1 的结果）：** 30 个验证场景跑完。带图的结果在 [phase1_results.ipynb](phase1_results.ipynb)，记录在[讨论稿](../../docs/mem/mem分支_问题定义与讨论稿_2026-11-10.md)第 22、23 节。

## 已定的约定（前三轮讨论）

这些是各环节共同的前提，改动任何一条都会影响多个环节。

- **记忆从视频构建。** 输入只有 RGB、简单深度和设备位姿；激光扫描、由扫描渲染的深度、扫描配准的位姿只用于制作训练标签和评测。
- **单目深度为主**，用设备 LiDAR 校正尺度；纯 LiDAR 作为对照。允许使用设备位姿（ARKit、Aria）。
- **动作词表**沿用 SceneFun3D 的 9 类：rotate、key_press、tip_push、hook_pull、pinch_pull、hook_turn、foot_push、plug_in、unplug。
- **人形只用于展示**，但"怎么接近、位置准、动作对"是展示的核心：affordance 是为之后的动作做准备。
- **不自采数据**；可以使用 SMPL-X 和仅限研究用途的公开数据。
- **阶段安排（2026-11-11 改为方案 B）：** S1（SceneFun3D 静态视频）做完重建对照后，先用一段 HD-EPIC 视频把事件层的最小版本跑通；然后是 S2（FunGraph3D 交互视频，检验交互能否帮助构建记忆）和 S3 的其余部分（HD-EPIC，检验事件结构能否帮助检索）。
- **流程确认之前不做正式训练和完整评测**；GPU 作业只通过服务器队列运行。

**第三轮讨论新增（2026-11-11）：**

- **记忆带时间，并记录实际发生的动作。** 检索可以拿一个具体的动作当线索。
- **不做稠密重建。** 不融合稠密几何，不建网格，不做逐场景优化；允许使用设备自带的位姿和单帧深度。三维从证据帧现算，得到的是相对位置。
- **精度分两个阶段。** 先"找到并走到跟前"（几十厘米以内），由记忆负责；再"碰到"（3 cm 以内），由证据帧里部件附近的信息负责，或者到了跟前重新看一眼。
- **存储用 implicit（隐式）的形式。** 第四轮讨论定为 L2，见下。

**第四轮讨论新增（2026-11-11）：**

- **记忆采用 L2 形式。** 每个关键帧的图（原图、深度、位姿、时间、affordance map），加上部件 token 和事件 token。
- **薄索引保持显式。** 时间、相机位姿、证据帧编号、token 在帧里的位置，以及事件里离散的部分（第几次、起止时间、挂在哪个部件上）。
- **几何量不存。** 接触点、法向、站立点在读取时从证据帧现算；评测照旧。
- **检索分两步。** 先用 affordance map 上有特点的内容结合传统语义匹配、搜索关键帧，再交给视觉语言模型。
- **两种特征都存。** 图文对齐的特征和自监督特征。

**第六轮讨论新增（2026-11-12）：**

- **主张：功能记忆。** 环境记忆按"能对什么做什么、做过什么"来组织和写入，而不是按帧、按物体或按融合出的几何。
- **目标会议：** 以 NeurIPS 为主，ICCV 到阶段 2 的关口再定。
- **人形展示降为补充材料里的演示**（取代上面"人形只用于展示"一条里"展示的核心"的说法）。
- **阶段 1 已批准：** 30 个验证场景、修改关键帧选取、重建对照，优先用 GPU（走队列）。规格见[讨论稿](../../docs/mem/mem分支_问题定义与讨论稿_2026-11-10.md)第 21 节。

## 总览（当前版本：L2 记忆，2026-11-11 第四轮讨论后）

记忆的形式已经定下来（L2）：**每个关键帧的图，加上部件 token 和一层很薄的显式索引。** 接触点等几何量不存，读取时从证据帧现算。检索分两步：先用 affordance token 和传统语义找关键帧，再交给视觉语言模型。

```text
写入（每段视频一次）
  输入        1 关键帧选取 ── 2 深度 ── 3 相机位姿
                ↓
  单帧理解    4 affordance map ── 5 部件运动 ── 14 交互检测
                ↓
  记忆        观察层  每个关键帧：原图、深度、位姿、时间、affordance map
              部件层  9 部件 token（两种特征 + 薄索引）── 6 跨帧关联
              事件层  15 事件 token 与状态时间线

读取（每个查询一次）
  查询        10 先找关键帧（affordance token + 传统语义）→ 再由视觉语言模型选出 token
                ↓
  现算几何    7 接触点 ── 8 法向、地面与站立点          只用证据帧和相邻几帧，不存
                ↓
  人形展示    11 末端姿态 ── 12 走近过程与画面

评测          13 准确度、成本、与重建对照
```

**每个环节都有一个真实数据上的示例**，见 [pipeline_examples.ipynb](pipeline_examples.ipynb)（编号相同，图可以直接拿去展示）。下表"示例里看到的"一栏来自那次运行：场景 421069 的一段视频，40 个关键帧，没有训练，没有用 GPU。**这些数字只说明每一步的行为，不是结果。**

| # | 环节 | 状态 | 示例里看到的 | 最主要的缺口 |
| --- | --- | --- | --- | --- |
| 1 | 关键帧选取 | 真实 | 取 10、20、40、80 帧时，看得到的部件是 12、24、29、34 个（共 38 个） | 交互前后的取帧，要等环节 14 |
| 2 | 深度 | 真实 | 单目深度全图中位误差 4.7 cm；部件处 11.2 cm | 小部件处的深度 |
| 3 | 相机位姿 | 真实 | 相机位置中位误差 7.2 cm，个别帧 89 cm | 同一个部件两次看到，位置相差中位 26 cm |
| 4 | affordance map | ORACLE | 40 帧里找出 62 个区域，覆盖 28 个真实部件；相机转得快的帧里标注和画面错位 | 没有模型；标注的对齐；相邻的同类部件会连成一个区域 |
| 5 | 部件运动 | ORACLE | 标注的运动轴画在帧上 | 没有预测方法；还没有进入 token |
| 6 | 跨帧关联 | 真实（初版） | 62 个 token 连成 39 个部件，真实部件 28 个；完全连对 14 个 | 主要靠位置，而位置误差和相邻部件的间距一样大 |
| 7 | 接触点 | 真实，读取时现算 | 中位误差 11.1 cm，落在部件上 13% | 小部件处的深度；多个 token 的证据还没合起来 |
| 8 | 法向、地面与站立点 | 真实，读取时现算 | 站立点在扫描里空闲 74% | 没看到的空间仍被当成空闲 |
| 9 | 部件 token | 真实（现成特征） | 每个 token 两个特征（384 维和 768 维）加薄索引；全部 token 的特征共 143 KB | 特征没有针对 affordance 训练 |
| 10 | 查询与检索 | 真实（现成模型） | 前 3 个关键帧里有正确部件：只用语义 83%，加上 affordance token 100%；视觉语言模型选对 18 个任务里的 10 个 | 小部件在整幅图里太小；动作类型还没用上 |
| 11 | 末端姿态 | 真实 | 用读出的几何量摆放人形 | 9 类动作只有伸手和伸脚两种姿态 |
| 12 | 走近过程与画面 | 真实 | 一段走近的短视频 | 不避障，脚会滑 |
| 13 | 评测 | 真实，单场景 | 接触点、站立点、关联、检索、存储大小、耗时 | 只有一个场景；重建对照组还没做 |
| 14 | 交互检测 | 做不了，只有示意 | SceneFun3D 的视频里没有人操作 | 需要带交互的视频 |
| 15 | 事件与状态时间线 | 做不了，只有示意 | 同上 | 同上 |

### 旧版总览 v1（2026-11-10 演示 notebook 里的流程，保留作对照）

```text
输入        1 关键帧选取 ── 2 深度 ── 3 相机位姿
              ↓
单帧理解    4 部件定位与动作类型（affordance map） ── 5 部件运动
              ↓
记忆构建    6 跨帧关联与合并 ── 7 接触点 ── 8 法向、地面与站立点 ── 9 条目内容
              ↓
查询        10 查询与检索
              ↓
人形展示    11 末端姿态 ── 12 走近过程与画面

评测        13 评测（贯穿所有环节）
```

| # | 环节 | 状态 | 最主要的缺口 |
| --- | --- | --- | --- |
| 1 | 关键帧选取 | 真实 | 38 个部件有 9 个没进记忆，原因没查清 |
| 2 | 深度 | 真实 | 部件处误差 11.2 cm，是全图的两倍多 |
| 3 | 相机位姿 | 真实 | 个别帧误差大，没有任何校正 |
| 4 | 部件定位与动作类型 | ORACLE | 没有模型，也没有批量制作训练标签的流程 |
| 5 | 部件运动 | ORACLE | 没有预测方法 |
| 6 | 跨帧关联与合并 | 缺 | 整个环节 |
| 7 | 接触点 | 真实 | 只用一帧；单目深度下只有 14% 落在部件上 |
| 8 | 法向、地面与站立点 | 真实 | 没看到的空间被当成空闲；不检查手是否够得着 |
| 9 | 条目内容 | 部分 | "动作结果"还没有自己的表示 |
| 10 | 查询与检索 | ORACLE | 没有真正的检索，也没有对照组 |
| 11 | 末端姿态 | 真实 | 9 类动作只有伸手和伸脚两种姿态 |
| 12 | 走近过程与画面 | 真实 | 不避障，脚会滑 |
| 13 | 评测 | 真实，单场景 | 只有一个场景；缺检索、运动和官方指标 |

## 总览的调整：加入时间和动作，不做稠密重建（2026-11-11）

这一节回应你对总览提出的两点。下面四格依次是：我对需求的理解、调整后的总览、相关研究、我的观点和需要你定的点。**状态（第三轮讨论后）：** 需求理解、"不做稠密重建"的说法、精度分两个阶段、阶段方案 B 已经由你确认，见本节末尾的决定表。总览 v2 和环节 14、15 要等存储形式定了再定，所以 **v1 的各环节正文仍然没有改动**。

### 我对两点需求的理解

**需求一：记忆要有时间，检索可以用实际发生过的动作当线索，尤其是第一视角视频。**

- 记忆里的每次观察都带时间。
- 视频里实际发生的动作作为事件存下来：什么时候、对哪个部件、做了什么、结果如何。
- 检索时可以拿一个具体的动作当线索，例如"我拿刀之前打开的那个抽屉"、"烤箱上次是什么时候打开的"、"水龙头现在开着吗"。
- 所以记忆里要有三样东西：**能做什么**（affordance，现在已有）、**做过什么**（事件，现在没有）、**状态怎么随时间变**（现在没有）。

**需求二：用 affordance map 和深度代替重建，得到相对的三维位置。**

- 不建全局三维模型：不融合点云，不建网格，不做逐场景优化。
- 三维信息只来自"某一帧的深度 + 这一帧上的 affordance map"。
- 得到的是相对位置：部件相对于看到它的那一帧相机，以及部件与部件之间。
- 目标是成本明显更低，同时三维任务仍然做得了：定位部件、接触点、站位、相对位置类问题。

**两点其实是同一个设计。** 场景会随动作改变，融合出来的静态三维模型装不下"变化"，带时间的关键帧观察天然可以。合起来就是：

> 记忆 = 带时间的关键帧观察，加上从中整理出的部件和事件；需要三维时，从证据帧现算。

**你的回复（2026-11-11）：** 理解正确。另外，记忆和系统的存储你更倾向 implicit（隐式）的形式。

### 总览 v2（提案）

**（第四轮讨论后：这一格的提案已经落实到最上面的"总览"里，这里保留作记录。）**

```text
输入        1 关键帧选取（均匀 + 动作前后） ── 2 深度 ── 3 相机位姿（只要求相近的关键帧之间可靠）
              ↓
单帧理解    4 affordance map：可能的动作 ── 5 部件运动 ── 14 交互检测：实际发生的动作（新增）
              ↓
记忆构建    观察层  每个关键帧：时间、位姿、深度、affordance map；不融合成全局模型
            部件层  6 跨帧关联与合并 → 7 接触点 → 8 站立点 → 9 条目内容        "能做什么"
            事件层  15 事件与状态时间线（新增）：何时、哪个部件、做了什么、前后状态  "做过什么"
              ↓
查询        10 空间 × 时间 × 动作 × 语义 → 部件 + 事件 + 证据帧；三维关系从证据帧现算
              ↓
人形展示    11 末端姿态 ── 12 走近过程与画面（也可以重放某个时刻发生过的动作）

评测        13 准确度 + 成本（与重建对照组比较）+ 时间类问题
```

**与 v1 相比，各环节的变化：**

| 环节 | 变化 |
| --- | --- |
| 1 关键帧选取 | 除了均匀取帧，在每次交互的前后各取一帧。接触前的一帧里部件没有被手挡住，接触后的一帧记录结果 |
| 2 深度 | 第一视角设备（Aria）没有 LiDAR，尺度改由设备定位给出的稀疏点来定 |
| 3 相机位姿 | 不再要求全程一致，只要求时间相近的关键帧之间可靠；隔得远的关系只保证方位和大致距离 |
| 6 跨帧关联与合并 | 还要把事件挂到部件上；同一个部件的外观会随状态改变（门开着和关着） |
| 7 接触点、8 站立点 | 以证据帧为参考来存（像素 + 深度），需要时再换算；是否空闲只看附近关键帧观察到的点 |
| 9 条目内容 | 增加时间字段（首次和最近一次看到、每次观察的时间）、当前状态、指向事件的链接；区分"预测的可能结果"和"观察到的结果" |
| 10 查询与检索 | 查询可以带时间和动作线索；增加一个对照：只有时间戳、没有动作结构的记忆 |
| 11、12 人形 | 除了"可以怎么做"，还能重放"当时是怎么做的" |
| 13 评测 | 增加成本指标和重建对照组；增加时间类问题 |
| 14 交互检测（新增） | 见文末环节 14 |
| 15 事件与状态时间线（新增） | 见文末环节 15 |

新增的两个环节编号接在 13 后面，避免改动已有编号；它们在流程中的位置以上图为准。

### 相关研究

**核实方式：** 三个检索 agent 逐条打开论文页面（arXiv、项目页或会议页），与之前的调研文档相同，只读摘要和表格，没有精读全文，也没有复现。标 ★ 的六篇 2026 年论文由我另外单独复核过摘要；摘要里没有写明、只来自正文的说法，标注"未复核"。之前调研里已有的工作直接沿用原链接。

**A. 不做稠密重建：在二维里找，再用深度提升到三维**

| 工作 | 做法 | 对我们的意义 |
| --- | --- | --- |
| [EgoLoc](https://arxiv.org/abs/2212.06969)（ICCV 2023） | Ego4D 的三维定位任务：二维检测 + 单目深度 + 相机位姿反投影，多帧加权平均；输出的是物体相对于提问那一帧相机的位移。成功率 87.1%，原基线 8.7% | 最接近"相对坐标"的先例。弱点：受位姿覆盖限制；深度换成随机值后成功率从 80.5% 降到 34.2%；物体被移动后失败 |
| ★ [R4DSG](https://arxiv.org/abs/2608.11017)（ACM MM 2026） | 长第一视角视频，只用 RGB；"相对 4D 场景图"记忆，条目按时间、地点、物体、相对变化来索引；明确不建全局对齐的世界模型。EgoLifeQA 的 255 题子集上比文本检索基线高 6.7 个点，"何时"类问题高 12.5 个点 | 与两点需求都最接近：相对三维 + 时间。但它是物体级的，没有功能部件、affordance 和动作类型 |
| [OSNOM](https://arxiv.org/abs/2404.05072)（3DV 2025） | 把物体提升到三维，离开视野后继续跟踪。单目深度要对齐到一个重建出的网格来定尺度，网格平均每段视频要算 5 小时 | 说明"提升"可行；但它靠重建来定尺度，我们用设备深度或稀疏点来定 |
| [Fun3DU](https://arxiv.org/abs/2411.16310)（CVPR 2025）；★ [UniFunc3D](https://arxiv.org/abs/2603.23478)（NeurIPS 2026） | SceneFun3D 上的功能部件：用视觉语言模型在几十帧里找部件，再投回点云做多视角投票。Fun3DU 的 AP25 为 33.3，三维网络基线 OpenMask3D 只有 0.4；UniFunc3D 到 51.0 | 支持"在二维里找小部件比三维网络好"。但它们都需要现成的点云来投票。**只用少量关键帧、不用点云的做法，没有找到** |
| [VLM-Grounder](https://arxiv.org/abs/2410.13860)（CoRL 2024） | 只用带位姿的图像序列做三维物体定位，不用重建的点云。ScanRefer 上宽松标准 51.6，点云方法 36.4；严格标准持平（32.8 对 32.7） | 粗定位可以超过点云方法，精定位只是持平 |
| [3D-Mem](https://arxiv.org/abs/2411.17735)（CVPR 2025）；[Mobility VLA](https://arxiv.org/abs/2407.07775)（CoRL 2024） | 3D-Mem 用少量图像快照代替三维地图做问答。Mobility VLA 把一段巡视视频的帧连成图，到了附近用图像匹配定位，再按相对位移移动 | "图像就是记忆"的先例；Mobility VLA 是"近处精确、远处只靠连接关系"的做法 |
| [Video-3D LLM](https://arxiv.org/abs/2412.00493)（CVPR 2025）对比 [VG-LLM](https://arxiv.org/abs/2505.24625)（NeurIPS 2025） | 前者给每个图像块加上由深度和位姿算出的位置，后者只用 RGB。ScanRefer：58.1／51.7 对 41.6／14.9 | 深度和位姿不能省 |
| [VGGT](https://arxiv.org/abs/2503.11651)（CVPR 2025）；[CUT3R](https://arxiv.org/abs/2501.12387) | 前馈式重建。VGGT 处理 100 帧约 3 秒，占 21 GB 显存；24 GB 的显卡只放得下 60–80 帧；物体大幅运动时会失败。CUT3R 可在线运行，长序列会漂移 | 重建已经不慢了。"省算力"必须用数字说话，见下一格第 3 点 |
| [ArtiPoint](https://arxiv.org/abs/2509.01708)（CoRL 2025） | 手出现时触发一段交互，跟踪点的运动得到铰接类型和轴；只用深度和里程计，不做全局重建。轴误差约 15–17°，位置误差 7 cm | 不重建也能从交互估计部件怎么动（环节 5） |
| [EPIC-Aff](https://arxiv.org/abs/2309.02120)（ICCV 2023） | 由旁白动词、物体掩码和相机位姿自动生成 affordance 标签，并累积成一张地图；3.9 万张图，20 类动词 | 最接近"第一视角的 affordance 地图"，但依赖运动恢复结构；可作为环节 4 的训练或评测数据 |

**B. 带时间和事件的第一视角记忆**

| 工作 | 做法 | 对我们的意义 |
| --- | --- | --- |
| [AMEGO](https://arxiv.org/abs/2409.10917)（ECCV 2024） | 在线、不训练：把手—物交互整理成片段（起止帧、哪只手、哪个物体），加上地点片段；之后不看视频只查记忆。自带基准 2 万多个问题，36.3% 对最好的字幕基线 23.6% | 事件层的现成做法（环节 14、15）。只有二维，没有部件、affordance 和三维 |
| ★ [Pandora](https://arxiv.org/abs/2603.28732)（BMVC 2025） | Aria 眼镜的第一视角数据 → 带铰接部件（抽屉、柜门）和包含关系的三维场景图，机器人据此取出柜子里的物品 | 最接近"第一视角 + 部件"的工作；没有时间检索和动作类型。正文称用手部交互更新部件状态、用设备位姿加估计深度（未复核） |
| ★ [ChronoGraph](https://arxiv.org/abs/2609.39665)（2026，预印本） | "功能 4D 场景图"：把作用在 affordance 部件上的动作和状态变化连起来，包括观察到的和预期的变化；配有基准 | 在"部件 + 动作 → 结果"上最接近。它是给视觉语言模型推理用的表示，针对短片段（未复核），不是长视频上可检索的持久记忆 |
| ★ [Ledger](https://arxiv.org/abs/2610.10538)（2026，预印本） | 第一视角视频上的持久三维物体记忆：位置、历史、描述；回答空间问题时不再看视频。摘要称 HD-EPIC 上从 29.7% 提高到 42.6% | 物体级；靠位置变化推断物体被移动，不用动作 |
| [GESTO](https://arxiv.org/abs/2608.10886)；[EGG](https://arxiv.org/abs/2510.18697)；[Embodied VideoAgent](https://arxiv.org/abs/2501.00358)；[Lost & Found](https://arxiv.org/abs/2411.19162) | 之前调研里已有：把带时间的交互事件写进场景图或物体记忆 | 物体级；EGG 去掉事件层后时间类问题从 0.78 降到 0.10 |
| [EM²Mem](https://arxiv.org/abs/2609.00551)（2026，预印本） | 以事件为锚点的记忆单元 | 250 个问题上：结构化事件字段 71.2，压平成字幕 65.6，直接给原始帧 68.0。结构有帮助，但幅度不大 |
| [HD-EPIC](https://arxiv.org/abs/2502.04144)（CVPR 2025） | 26,650 个问题；真值有 59,454 个动作区间、1.99 万条物体搬移轨迹、带 413 个固定设施的数字孪生 | S3 的评测依据。有"和哪个设施交互了几次"、"物体在哪"这类问题；**没有找到部件状态（开或关）的标注** |

**C. 可执行的 affordance 和交互检测（环节 4、5、14）**

| 工作 | 做法 | 对我们的意义 |
| --- | --- | --- |
| [3DOI](https://arxiv.org/abs/2305.09664)（ICCV 2023） | 单张图加一个查询点 → 能不能动、转动还是平移、轴（二维）、推还是拉、接触位置、掩码、深度；1 万张图，其中 2 千张来自 EPIC-Kitchens | 输出形式和我们的条目最接近，可作为环节 4、5 的起点或对照 |
| [VRB](https://robo-affordances.github.io/)（CVPR 2023）；[RAM](https://arxiv.org/abs/2407.04689)（CoRL 2024） | 从人类视频学"接触点 + 接触后的运动方向"。RAM 用接触点处的深度提升到三维，方向取自局部法向 | 与我们环节 7、8 的做法相同：接触点深度加局部法向，不需要重建 |
| [A3VLM](https://arxiv.org/abs/2406.07549) | 单张 RGB → 可操作部件的框、三维轴、类型（转动或平移） | 环节 5 可以从单帧预测轴的一个证据 |
| [100DOH](https://arxiv.org/abs/2006.06669)（CVPR 2020）；[Hands23](https://proceedings.neurips.cc/paper_files/paper/2023/hash/612a7948f3294a02a63d970566ca8536-Abstract-Conference.html)（NeurIPS 2023） | 现成的手—物接触检测器，逐帧输出手的框、左右、是否接触、物体框；代码为 MIT 许可 | 环节 14 可直接用。它们不给起止时间和接触像素，要自己整理（AMEGO 的做法） |

**从这些文献得到的五点：**

1. **在二维里找部件再提升，在功能部件上已经超过三维网络。** 但现有方法都把结果投回现成的点云做多视角投票。只用少量关键帧、不用点云，是空位。
2. **深度和位姿不能省。** 相对三维可行（EgoLoc、R4DSG）；隔得远的关系需要显式的位姿，视觉语言模型自己推不出来。
3. **带事件的记忆在时间类问题上有增益。** 但没有人在同一份记忆上干净地比较过"只有时间戳"和"时间戳 + 动作结构"。也有反例：一些结构化记忆不如直接把帧交给大模型。
4. **没有找到把"能做什么"（部件级 affordance）和"做过什么、何时"（观察到的事件）存在同一份可检索记忆里的工作。** 最接近的是 ChronoGraph 和 Pandora，各缺一半。
5. **"看人操作能更好地找到小部件"没有直接证据。** 现有的间接证据是混合的。这是 S2 要检验的假设，也可能不成立。

### 我的观点和需要你定的点

**1. 把两点需求合成一个主张。** 带时间的关键帧观察就是记忆；部件层和事件层是从观察里整理出来的索引；三维在查询时从证据帧现算。相近的工作各占一部分：R4DSG（相对三维 + 时间，物体级）、AMEGO（交互事件，只有二维）、Pandora 和 ChronoGraph（部件 + 状态，没有长视频检索）、Fun3DU 一类（在二维里找功能部件，但需要点云）。同时做到"功能部件级、不用点云、带事件时间线、可检索"的，我没有找到。

**2. "不重建"需要一个精确的说法，否则会被质疑。** 建议定为：不融合稠密几何，不建网格，不做逐场景优化；允许使用设备自带的位姿和单帧深度。要注意两点：

- 设备位姿本身来自设备内部的定位，所以准确的说法是"不做稠密重建"，而不是"不用任何三维"。
- 没有设备位姿的视频（普通的 EPIC-KITCHENS、网络视频）不在这个说法之内，除非另外估计位姿。

**3. "省算力"要用数字证明，而且对手已经很快。** VGGT 处理 100 帧只要约 3 秒。所以我建议把理由放在三处，并在评测里加一个重建对照组：

- **长视频：** 记忆的大小随部件和事件的数量增长，不随帧数和场景体积增长；前馈式重建一次只放得下几十到一两百帧。
- **场景会变：** 融合出的模型假设场景静止，动作一发生它就过时了。
- **小部件：** 融合会抹掉小部件；在 SceneFun3D 上，二维方法已经超过三维网络。
- **重建对照组：** 同一段视频、同样的标签，一边先融合点云再读接触点，一边只用关键帧，比较精度、耗时和存储。这个现在就能在场景 421069 上用 CPU 试。

**4. 相对坐标：建议"近处精确，远处只保证方位"。**

- 同一帧里的两个部件，相对位置直接由这一帧的深度给出，不受位姿误差影响。
- 不同帧之间要靠位姿，隔得越久越不准（ARKit 个别帧差 89 cm）。
- 因此接触点、站位、人形这些近处的量只依赖证据帧本身；远处的关系只保证方位和大致距离，到了跟前再重新定位。EgoLoc 和 Mobility VLA 都是这样做的。
- 需要一个试验来量化：两个部件相对位置的误差，随它们所在关键帧的时间间隔怎么变。

**5. 精度建议分两档。** 现在用单目深度，只有 14% 的接触点落在部件上。文献里小部件同样是难点：UniFunc3D 在占画面不到 0.1% 的部件上明显变差。建议把目标拆开：

- **找到并走到跟前**（几十厘米以内）：由记忆负责。
- **碰到**（3 cm 以内）：由证据帧里部件附近的信息负责（环节 2、7 的改进），或者到了跟前重新看一眼。

这会改变环节 2、7 的要求，也影响人形画面"位置准"的说法，需要你同意。

**6. 事件从哪里来。** 现成的手—物检测器是逐帧的，起止时间和接触位置要自己整理。接触的那一刻手会挡住部件，所以用接触前的一帧定位部件，用接触后的一帧记录结果。HD-EPIC 有动作区间和物体搬移的真值，但没有部件状态的标注；这部分真值要自己补，或者用有铰接状态的数据（Hoi!、RealFun4D）。

**7. 对阶段安排的影响。** S1 的 SceneFun3D 视频里没有人操作，事件层是空的：只能有"何时看到"，没有"何时做过"。带动作的时间检索要到有交互的数据上才能做。两个选择：

- **A：** 保持 S1 → S2 → S3。S1 只把部件层和"不重建"的对照做扎实。
- **B：** S1 做完重建对照后，先拿一段 HD-EPIC 视频把事件层的最小版本跑通，相当于把 S3 的一小部分提前。

我倾向 B：事件层决定条目和查询怎么设计，越早看到真实数据越好。顺序由你定。

**五件事的决定（2026-11-11，第三轮讨论）：**

| # | 问题 | 你的决定 |
| --- | --- | --- |
| 1 | 对两点需求的理解 | 正确。补充：记忆和系统的存储更倾向 implicit（隐式）的形式 |
| 2 | "不做稠密重建"的说法（第 2 点） | 接受 |
| 3 | 精度分两档（第 5 点） | 可以，分两个阶段 |
| 4 | 阶段选 A 还是 B（第 7 点） | B |
| 5 | 总览是否改用 v2，环节 14、15 是否保留 | 第三轮未定。第四轮你定了存储形式并要求按它改写文档，所以总览已改为 L2 版本，环节 14、15 保留（方案 B 需要它们）。第五轮（2026-11-12）你确认了这一条 |

第 3 条我的理解是：先做"找到并走到跟前"，再做"碰到"，两者的指标分开报告。理解得不对请直接改。

存储形式的讨论在下面"记忆的存储形式"一节。

### ✏️ 总览 · 你的更新

（写在这一行下面）

## 记忆的存储形式：implicit（隐式）怎么做（2026-11-11，讨论中）

你在第三轮讨论里说：记忆和系统的存储更倾向 implicit 的形式；具体怎么存，你打算再读一些文章。这一节把"别人是怎么存的"摆出来，供你对照着读。下面四格依次是：implicit 指什么；参考文章的简介；文献里隐式和显式的对照证据；放到我们的流程里怎么做、我的建议和要你定的点。**这一节只是讨论材料，没有改任何环节的做法。**

### "implicit" 在文献里指五种不同的东西

同一个词，各篇文章指的不是一回事。区别在两处：**记忆里的一个单元是什么**，以及**能不能单独取出某一条**。

用一件事举例（示意，不是数据里真有的事件）：**烤面包机的旋钮可以拧；10:32 有人拧过它。** 它在各种记忆里分别是这样的：

| 形式 | 这件事在记忆里是什么 | 怎么读出来 | 代表工作 |
| --- | --- | --- | --- |
| 显式条目（我们现在的做法） | 一条记录：动作 = rotate，接触点 = (x, y, z)，法向，站立点，证据帧 = 第 17 帧 | 按字段查 | 演示 notebook |
| 显式文字或图 | 一句话"10:32 拧了烤面包机的旋钮"，或者图上"人 → 拧 → 旋钮"的一条边 | 文字检索，再交给语言模型 | R4DSG、AMEGO |
| ① 记忆库 | 几个特征向量（旋钮那一小块画面的特征），每个都带着"哪一帧、几点、相机在哪" | 拿查询的特征去比相似度或做注意力，再按时间和位置筛 | SAM 2、XMem、WorldMem、3DLLM-Mem、ReKV |
| ② 带位置编码的 token | 关键帧切成小块，每块的特征上叠加了"这一块在三维里的位置"的编码；旋钮只是其中几块，没有单独的一条 | 全部交给一个语言模型去读 | Video-3D LLM、LLaVA-3D、LLaVA-4D |
| ③ 循环状态 | 一组数量固定的状态向量，每来一帧整体更新一次；旋钮的信息混在里面 | 只能让模型解码，取不出"旋钮那一条" | CUT3R |
| ④ 特征场 | 一个函数：给一个三维位置（和时间），返回那里的特征；旋钮就是特征像"可以拧"的那片区域 | 在空间里取点，与查询比相似度 | CLIP-Fields、D3Fields、4D LangSplat |
| ⑤ 直接存图 | 拍到旋钮的那几帧原图，加上时间 | 需要时让视觉语言模型重新看图 | 3D-Mem |

**三点观察：**

- **多数"隐式"记忆仍然有显式的索引。** ①、②、④、⑤ 都留着时间、相机位姿或位置，只有内容是特征。完全没有索引的只有 ③。
- **"隐式"和"不做稠密重建"不冲突。** ② 就是不重建的做法：用单帧深度和位姿算出每一小块的位置，再编码进特征里。
- **有两类做法与已定的约定冲突。** 需要逐场景优化的特征场（CLIP-Fields、4D LangSplat）违反"不做逐场景优化"；把记忆放进模型权重里的做法同理。不需要优化的特征场（D3Fields）不冲突。

### 别人怎么存：参考文章的简介

每篇用同一个格式：先画出"写入、记忆、读取"，再列几条要点，最后一条是它对我们意味着什么。数字都出自论文正文或表格。

#### ① 记忆库：SAM 2 和 XMem（在视频里一直认出同一个东西）

[SAM 2](https://arxiv.org/abs/2408.00714)、[XMem](https://arxiv.org/abs/2207.07115)（ECCV 2022）

```text
写入   每处理完一帧 ──→ 这一帧的特征图和预测出的掩码融合成一张"记忆特征图"，再加一个代表这个物体的短向量
记忆   SAM 2：最近 6 帧，加上被人点过提示的帧，先进先出
       XMem：三层。每帧都更新的隐状态；最近几帧的高分辨率特征；由常用特征压成的少量原型（长期记忆）
读取   新一帧的特征对记忆做注意力 ──→ 得到"这个物体现在在哪"的掩码
```

- **一个单元：** 一帧里这个物体的特征图（SAM 2 是 64 维），加一个 256 维的物体向量。没有坐标，没有文字；SAM 2 只给最近几帧加了时间位置编码。
- **记忆变长怎么办：** SAM 2 直接丢掉最旧的。XMem 把工作记忆里用得最多的特征压成原型，放进长期记忆；长期记忆满了，再淘汰最少用到的。
- **几种记忆的对照：** XMem 三层都用是 89.8，去掉工作记忆、只靠每帧更新的隐状态是 38.7（Long-time Video）。SAM 2 试过把记忆先过一遍循环网络再存，没有提升，所以直接存特征。
- **要注意：** 长时间遮挡或画面拥挤时会跟丢、认错。两者都需要有人先在某一帧给出提示。
- **对我们：** 这是环节 6（跨帧认出同一个部件）现成的隐式做法。XMem "常用的留下、不常用的淘汰"是一种整理长期记忆的规则。

#### ① 记忆库：WorldMem（每个单元带位姿和时间）

[WorldMem](https://arxiv.org/abs/2504.12369)（NeurIPS 2025）

```text
写入   每生成一帧 ──→ 存下（这一帧的隐特征，相机位姿，时间戳）；全部保留，不合并
读取   第一步 挑：按"与当前视野重叠多少"减去"时间隔了多久"打分，去掉彼此太像的，留下 8 个单元
       第二步 读：当前帧的特征对这 8 个单元做注意力；位姿和时间先换算成相对当前帧的量，再编码进去
```

- **一个单元：** 内容是一帧的隐特征（隐式），索引是位姿和时间戳（显式）。600 个单元约 21 MB。
- **时间有什么用：** 同一个地方在不同时刻是两条记忆，所以放下的东西还在，麦子会长高。去掉时间条件，PSNR 从 25.12 降到 23.17。
- **怎么挑很重要：** 随机挑是 18.32，按视野重叠和时间挑是 23.12，再去掉重复的是 23.98。位姿用相对量编码（23.98）好于用绝对量（23.63）。
- **要注意：** 它是视频生成模型的记忆，位姿来自模拟器或标注；记忆随帧数线性增长；视野重叠判断不了遮挡。[VMem](https://arxiv.org/abs/2506.18903)（ICCV 2025）为此改用带遮挡的几何索引来挑帧，14.82 对 13.11。
- **对我们：** 最接近你说的"带 4D 时间戳"的隐式记忆。"显式的位姿和时间做索引，隐式的特征做内容，读的时候换算成相对量"可以直接借用，也和"相对坐标"的想法一致。

#### ① 记忆库：3DLLM-Mem（具身智能体的长期记忆）

[3DLLM-Mem](https://arxiv.org/abs/2505.22657)（NeurIPS 2025）

```text
写入   眼前看到的 ──→ 一组带三维位置编码的特征，留在模型的上下文里（工作记忆）
       离开这个房间 ──→ 这组特征加上时间编码，存进记忆库（情景记忆）
       回来后，如果自己动过这里的东西 ──→ 覆盖这一条
读取   用工作记忆的特征做查询，对记忆库做注意力；取回的特征与工作记忆拼在一起，交给语言模型
```

- **一个单元：** 一次观察的一组特征（就是下面 LLaVA-3D 的三维小块特征），加上时间编码。
- **读法的对照：** 用注意力融合，平均成功率是 37.6（域内）和 32.1（域外）；"先检索出几条，再放进上下文"是 22.3 和 15.6；只留最近的记忆是 21.1 和 13.7。
- **要注意：** 只处理智能体自己造成的变化；需要训练（约 2.6 万条轨迹）；动作是模拟器里预先定义的高层动作。
- **对我们：** 结构上最像"观察层 + 时间"的隐式长期记忆。它的读取是训练出来的注意力，不是相似度检索。

#### ② 带位置编码的 token：Video-3D LLM 和 LLaVA-3D

[Video-3D LLM](https://arxiv.org/abs/2412.00493)（CVPR 2025）、[LLaVA-3D](https://arxiv.org/abs/2409.18125)（ICCV 2025）

```text
写入   关键帧 ──→ 图像编码器 ──→ 每帧几百个小块的特征
       深度 + 位姿 ──→ 每个小块在世界里的坐标 ──→ 位置编码 ──→ 加到小块的特征上
记忆   8 到 32 帧的"带位置的小块特征"，按帧的顺序排好；没有点云，没有物体列表
读取   和问题一起交给语言模型；要输出三维框时，另接一个解码头
```

- **一个单元：** 一个图像小块的特征，加上它的三维位置编码。坐标本身不存，只以编码的形式进入特征。
- **需要什么：** 每帧的 RGB、深度和相机内外参；不需要点云，不需要逐场景优化。两者都要微调整个视频语言模型（8 到 16 块 A100）。
- **位置编码有多大用：** 去掉它，需要指出"在哪里"的任务明显变差（Scan2Cap：Video-3D LLM 从 83.8 降到 31.0，LLaVA-3D 从 84.1 降到 29.7），普通问答几乎不变（ScanQA 30.1 对 30.0）。
- **要注意：** 它不是长期记忆，每个场景取几十帧一次读完；没有时间，也不处理场景变化。让语言模型直接写出三维框效果很差（LLaVA-3D：7.8，接解码头后 50.1）。Video-3D LLM 做定位时仍然用了点云上的物体候选框。
- **对我们：** 这是"用深度和位姿代替重建"最直接的隐式写法。但 token 是按整帧均匀切的，小部件占不到一个小块。

#### ② 加上更新规则和分层：Dynam3D

[Dynam3D](https://arxiv.org/abs/2505.11383)（NeurIPS 2025）

```text
写入   新的一帧（RGB-D + 位姿）──→ 576 个小块特征，各带世界坐标
       先删旧：在新画面的视野里、却挡在新测到的深度前面的旧特征点，全部删掉
       再合并：每个二维分割块与最近的几个已有实例比较，一个小网络判断是不是同一个；是就合并，不是就新建
记忆   三层：小块特征点（带坐标）→ 实例 token（一个物体一个向量）→ 区域 token（几立方米一个向量）
读取   三层 token 和指令一起交给语言模型，输出下一步怎么走
```

- **一个单元：** 最底层是带坐标的小块特征；上面两层是由它们汇总出来的向量。
- **场景变化：** 靠"先删旧再写新"。记忆里只有最新状态，没有时间戳。
- **结果：** R2R-CE 成功率 52.9；去掉实例和区域两层降到 45.7。目标被人挪动的实机测试成功率 45%，只存视频帧的 NaVid 是 20%。每步 649 ms，其中更新记忆 83 ms。
- **代价：** 表示模型和导航模型各训练 8 到 9 天。
- **对我们：** 三层结构对应我们的观察层和部件层；合并规则就是环节 6。但它删掉了旧状态，而我们要回答"之前是什么状态"，所以旧的不能删，只能标上时间。

#### ③ 循环状态，以及它的问题：CUT3R 和 Point3R

[CUT3R](https://arxiv.org/abs/2501.12387)（CVPR 2025）、[Point3R](https://arxiv.org/abs/2507.02863)（NeurIPS 2025）

```text
CUT3R    记忆 = 768 个状态向量，数量固定
         写入：每来一帧，图像特征和状态互相做注意力，状态整体更新一次，旧内容被悄悄覆盖
         读取：同一次计算里读出这一帧的三维点和相机位姿；取不出任何单独的一条

Point3R  记忆 = 一批"指针"，每个指针 = 一个明确的三维位置 + 一个特征向量
         写入：新指针离已有的指针很近就合并（新的盖掉旧的），否则新增
         读取：当前帧的特征对所有指针做注意力，三维位置以位置编码的形式参与
```

- **对照：** 在长序列上，Point3R 的重建误差是 0.071，CUT3R 是 0.238（7-Scenes）。Point3R 的摘要直接写道：隐式记忆容量有限，会丢失较早帧的信息。CUT3R 的作者也说明，序列很长时会漂移。
- **另一个极端：** [StreamVGGT](https://arxiv.org/abs/2507.11539)（ICLR 2026）把每一帧的 token 全部缓存下来，精度高，但 200 帧就要 25 GB 显存。
- **要注意：** Point3R 的指针随探索范围增长；合并时新的盖掉旧的，同样不保留历史。
- **对我们：** 纯状态的记忆最"隐式"，但放不下长视频，也答不了"什么时候"。给特征配一个显式的地址（位置、时间），是这条线上后续工作的主要改进方向。

#### ④ 按需计算的特征场：D3Fields

[D3Fields](https://arxiv.org/abs/2309.16118)（CoRL 2024）

```text
写入   每个相机当前的一帧 ──→ DINOv2 特征图 + 实例掩码图 + 深度图（只存这三张图）
读取   给一个三维点 ──→ 投影到每个相机 ──→ 取那个像素的特征，按"这个点在该视角是否可见"加权平均
       ──→ 返回这个点的特征，以及它属于哪个物体
```

- **一个单元：** 没有持久的三维结构。"场"只是一个按需计算的函数，存下来的是每个视角的二维图。
- **需要什么：** 固定并标定好的多视角 RGB-D；不训练，不做逐场景优化。
- **结果：** 4 个相机建场 0.17 秒，需要逐场景训练的 F3RM 是 88 秒（A6000）。
- **要注意：** 每一步都用当前画面重算，不存历史，没有时间。
- **对我们：** 这正是"需要三维时从证据帧现算"的隐式版本：记忆就是每帧的特征图和深度图，查询时再投影。不同之处是我们的各个视角来自不同时刻，而且相机在动。

#### 第一视角：Ego-Topo

[Ego-Topo](https://arxiv.org/abs/2001.04583)（CVPR 2020）

```text
写入   每来一帧 ──→ 与每个已有区域比"是不是同一个地方"（学出来的相似度）
       像 ──→ 并入这个区域，记一次"到访"（起止帧）；不像 ──→ 新建一个区域
       再记一条从上一个区域走到这里的边
记忆   一张图：节点是活动区域，节点里是历次到访的视频片段；边是走动的先后
读取   节点的特征 = 历次到访片段特征的平均；用来预测这个区域能做什么、接下来会做什么
```

- **一个单元：** 一个区域，加上它的全部到访片段。图的结构和到访时间是显式的，"是不是同一个地方"和节点的内容是隐式的。
- **需要什么：** 只要 RGB，不用深度、位姿和重建。文中报告 SLAM 只能注册 EPIC-Kitchens 44% 的帧（EGTEA+ 是 6%）。
- **结果：** 区域 affordance 的 mAP，EPIC 上 29.4（SLAM 网格 26.6），EGTEA+ 上 51.6（SLAM 网格 41.8）。
- **要注意：** 粒度是区域，到不了部件；不记录物体的状态变化。
- **对我们：** 这是第一视角视频里"不靠重建、按活动组织空间"的先例。节点里存的是带时间的到访片段，和我们的"部件 + 事件时间线"是同一种结构，只是粒度粗。

#### affordance 记忆：Robo-ABC

[Robo-ABC](https://arxiv.org/abs/2401.07487)（ECCV 2024）

```text
写入   人操作物体的视频 ──→ 找到手刚接触物体的那一帧 ──→ 取手和物体交叠处的几个点
       ──→ 换到附近一帧没被手挡住的清晰画面 ──→ 存下（物体的裁剪图，接触点的像素位置）
读取   新物体的图 ──→ 按 CLIP 相似度取回最像的几条 ──→ 用扩散模型的特征逐点找对应，把接触点搬到新图上
```

- **一个单元：** 一张裁剪图加几个接触点的像素位置。特征不存，查询时现算。
- **需要什么：** 写入只要 RGB 视频，读取只要一张图；全部用现成模型，不训练。
- **结果：** 接触点成功率 60.7%，VRB 是 26.9%；真机抓取 85.7%。
- **要注意：** 记忆是跨场景的经验库，不是某个场景的记忆；没有时间。
- **对我们：** 写入步骤就是环节 14 到 15 要做的事：接触那一帧被手挡住，就换到邻近的清晰帧。它也说明"存图，读的时候再算特征"可行。

#### 其他读过的文章（每篇一行）

| 文章 | 记忆里存的是什么 | 值得看的一点 |
| --- | --- | --- |
| [Scene Memory Transformer](https://arxiv.org/abs/1903.03878)（CVPR 2019） | 每一步的观察存成一个向量，带位姿和时间，只追加 | 最早的"全都存下，读的时候再整理"；好于 LSTM 的循环状态（474.6 对 418.1） |
| [MA-LMM](https://arxiv.org/abs/2404.05726)（CVPR 2024） | 每帧的图像特征；超出上限时，把最像的相邻两帧取平均 | 合并（63.0）好于丢最旧的（61.3）；没有记忆是 48.3 |
| [ReKV](https://arxiv.org/abs/2503.00540)（ICLR 2025） | 语言模型读过每一帧之后的内部缓存，全部存到内存和硬盘 | 不训练；提问时取回最相关的 64 帧；每小时视频占 18.8 GB |
| [MemoryVLA](https://arxiv.org/abs/2508.19236)（ICLR 2026） | 每一步 256 个感知 token 和 1 个语义 token，带时间编码 | 机器人策略里的记忆库；去掉时间编码，71.9 降到 69.8 |
| [3D-Mem](https://arxiv.org/abs/2411.17735)（CVPR 2025） | 原图，每张图代表一簇一起出现的物体 | 直接存图，提问时让视觉语言模型看图；好于场景图 |
| [LLaVA-4D](https://arxiv.org/abs/2505.12253)（ICLR 2026） | 特征上叠加 (x, y, z, t) 的编码 | 只加位置 88.6，只加时间 82.7，都加 93.5；但深度和位姿来自重建 |
| [RNR-Map](https://arxiv.org/abs/2303.00304)（CVPR 2023） | 俯视网格，每格一个 32 维特征，是落在这一格的像素特征的平均 | 不需要逐场景优化，建图每秒 92 帧；旧的里程误差改不回来 |
| [MTU3D](https://arxiv.org/abs/2507.04047)（ICCV 2025） | 每个物体一条：三维框、掩码、两个特征向量 | 显式的框加隐式的特征；不需要先重建点云 |
| [CLIP-Fields](https://arxiv.org/abs/2210.05663)（RSS 2023） | 一个网络：输入三维位置，输出特征 | 每个场景要单独训练；不处理变化 |
| [4D LangSplat](https://arxiv.org/abs/2503.10437)（CVPR 2025） | 每个高斯带 3 个"状态原型"特征，随时间在它们之间过渡 | 状态变化的一种隐式表示；每个场景要优化 24,000 步 |
| [4DLangVGGT](https://arxiv.org/abs/2512.05060)（预印本） | 不存场景模型，前向一次得到每个像素随时间变化的特征 | 把上一行改成不需要逐场景优化 |
| [EgoEnv](https://arxiv.org/abs/2207.11365)（NeurIPS 2023） | 查询时刻前后 64 帧，每帧一个向量 | 第一视角；在模拟环境里训练，让特征能预测周围有什么 |
| [Interaction Hotspots](https://arxiv.org/abs/1812.04558)（ICCV 2019）、[UAD](https://arxiv.org/abs/2506.09284)（ICRA 2025） | 什么都不存：affordance 在网络权重里，给一张图现算热力图 | affordance map 本身的两种做法 |

**核实方式：** 正文的方法和实验部分由检索 agent 打开全文阅读并摘录原句；每篇的摘要我另外逐篇复核过（OpenEQA 除外，agent 读的是会议论文的 PDF）。上面的数字出自 agent 摘录的正文和表格，我没有逐个对照原表。标"预印本"的没有经过同行评审。

### 隐式还是显式：文献里的对照证据

没有哪一种形式在所有任务上都赢。下表每一行都是同一个基准上的对照。

| 比的是什么 | 结果 | 出处 |
| --- | --- | --- |
| 原始帧，对文字记忆（具身问答） | 直接给帧 49.6；帧的文字描述 43.6；场景图的文字 36.5 到 38.9；不看图 33.5 | [OpenEQA](https://github.com/facebookresearch/open-eqa)（CVPR 2024）；给帧的一行只测了 500 题 |
| 存图，对场景图（边探索边问答） | 52.6 对 47.2 | 3D-Mem |
| 纯状态，对带地址的特征（流式三维重建） | 误差 0.238 对 0.071 | CUT3R 和 Point3R |
| 纯状态，对记忆库（视频里跟踪物体） | 38.7 对 89.8 | XMem 的消融 |
| 特征、文字、状态三种记忆（机器人，同一个模型，同样的记忆预算） | 过去帧的特征 44.51；文字子目标 32.70；循环状态 22.35；把文字换成标准答案是 84.08 | [RoboMME](https://arxiv.org/abs/2603.04639)（ICML 2026） |
| 文字和关键帧合用（机器人） | 只有文字 65.4；只有关键帧 72.6；两者都有 93.4 | [UniMem](https://arxiv.org/abs/2608.22869)（预印本） |
| 反例：文字赢（机器人，计数和进度类任务） | 文字记忆 85.7；关键帧 72.9；隐式记忆库 39.5 | [HeLM](https://arxiv.org/abs/2611.02660)（预印本） |
| 反例：状态式记忆赢（机器人） | 256 个循环更新的隐式 token，先经过专门的预训练：72.8；按规则选帧 67.8；文字加关键帧最高 42.0 | [Imprint](https://arxiv.org/abs/2610.14406)（预印本） |
| 位置要不要显式给 | 加上位置编码，需要指出位置的任务从 31.0 升到 83.8；不给深度和位姿、让模型自己从 RGB 里学，ScanRefer 是 41.6 对 58.1 | Video-3D LLM；VG-LLM（见上一节的相关研究） |
| 压缩时合并还是丢弃（第一视角） | 按重要性丢弃 45.12；合并取平均 41.87 | [EgoStream](https://arxiv.org/abs/2605.31557)（预印本） |

**我读出来的四点：**

1. **内容用特征或原图，好于先翻译成文字或场景图。** 外观、空间细节和运动尤其如此。文字的弱点主要在"写对"：RoboMME 里把文字换成标准答案，成绩从 32.70 升到 84.08。
2. **文字在计数和离散的进度上更好，也更省。** "第几次"、"做没做过"这类信息，用显式的记录更可靠。把两者合起来的 UniMem 明显好于只用其中一种。
3. **完全没有地址的状态式记忆，在多数对照里是最差的一档**（三维重建、视频分割、RoboMME）。有一个反例：Imprint 的状态式记忆经过专门的预训练后成绩最好，作者同时说明它比显式记忆难解释、难分析失败原因。
4. **压缩要小心，取平均会抹掉细节。** 按相似度合并好于直接丢最旧的（MA-LMM、MemoryVLA），但在第一视角的细节问题上，按重要性丢弃又好于合并。把特征平均进网格也是同样的道理。这和"融合会抹掉小部件"是同一个问题。

**没有找到的：**

- 把第一视角视频里人和物体的交互事件存成隐式特征的长期记忆。找到的第一视角记忆都是显式的文字或图。
- 不需要逐场景优化、为整个场景建立的 affordance 特征场。

每组只用了六个左右的检索词，所以只能说没有找到，不能说没有。

### 放到我们的流程里：三种隐式程度，和我的建议

**你的想法里已经有一半是隐式的。** affordance map 本身就是隐式表示：它是每个像素上的分数或特征，不是一条条记录。真正要定的只有一件事：**从图上抽出来的东西，是写入时就算好、存成字段，还是只存图和特征、读取时再算。**

按隐式的程度可以分三档：

```text
L1 只存图
   存：每个关键帧的原图、深度、affordance 图（或它的特征图），加上时间和位姿。没有部件条目。
   读：在所有关键帧的图上找；找到后用那一帧的深度现算三维。
   相近的做法：D3Fields、3D-Mem、ReKV

L2 图 + token
   存：在 L1 上加一层索引。affordance 图上每个高分区域汇成几个特征向量（部件 token），
       每次交互的前后各汇成特征向量（事件 token），各自带时间和证据帧编号。
   读：先在 token 里按相似度找，再按时间和位置筛，最后回到图上现算三维。
   相近的做法：SAM 2、XMem、WorldMem、Dynam3D 的实例层

L3 token 交给模型读
   存：和 L2 一样，或者整帧的 token。
   读：把记忆 token 直接送进语言模型或动作模型，读取的能力靠训练得到。
   相近的做法：Video-3D LLM、3DLLM-Mem、MemoryVLA
```

| | L1 只存图 | L2 图 + token | L3 交给模型读 | 现在的显式条目 |
| --- | --- | --- | --- | --- |
| 要不要训练 | 不要 | 不要，用现成特征 | 要，数据量大 | 不要 |
| 长视频 | 每次查询要扫所有关键帧 | 可以，先查 token | 受模型上下文长度限制 | 可以 |
| 能不能逐环节检查 | 能，直接看图 | 能，token 指回证据帧 | 难 | 最容易 |
| 回答"什么时候" | 只有"何时看到" | 可以，事件 token 带时间 | 取决于训练 | 可以 |
| 接触点等几何量 | 读取时现算 | 读取时现算 | 要另接解码头 | 写入时算好 |

**我的建议：S1 先做 L1；到长视频和事件时升到 L2；L3 等流程确认、可以训练之后再考虑。**

- L1 和 L2 都符合已定的约定：不做稠密重建，不做逐场景优化，现在不训练。
- L1 是你最初想法最直接的实现："affordance map + 深度"就是记忆本身。在场景 421069 上，它可以和现在的显式条目、和重建对照组用同一套指标比较。
- L2 的索引只为两件事：长视频里不必每次扫全部关键帧；事件要能按时间查。
- 文献支持"薄的显式索引 + 隐式的内容"：WorldMem 用位姿和时间做索引，Point3R 用位置做地址，SAM 2 直接存特征而不过循环网络。纯状态式的记忆在多数对照里是最差的一档。

**建议保持显式的部分：**

- 时间戳、相机位姿、证据帧编号、token 在帧里的位置。它们是索引，数据量很小。
- 事件里离散的部分：第几次、起止时间、挂在哪个部件上。文献里这类信息用显式的记录更可靠。
- 其余都可以是隐式的：部件是什么、能怎么用、现在是什么状态、动作的结果。

**隐式带来的四个代价：**

1. **看不见，不好评测。** 每个环节要配一个"读出"：从特征和证据帧里读出接触点、动作类型、状态。评测指标照旧，只是作用在读出的结果上。读出的结果不存，或者只当缓存。
2. **读取端迟早要训练。** 相似度检索不用训练。动作类型、运动轴、状态这些，要么训练一个小的读出头，要么让视觉语言模型看证据帧。LLaVA-3D 的经验是让语言模型直接写三维坐标很差（7.8 对 50.1），几何量要靠专门的读出。
3. **存储不一定更小。** 场景 421069 的激光扫描是 133 MB，40 个关键帧的原图约 13 MB（按数据集里的文件大小）。现在的显式条目是几十 KB 的量级（我的估算，不含用于检查站位的观察点）。如果每个关键帧再存一张 37×37、1024 维的特征图，40 帧约 110 MB（我的估算，半精度）。所以特征要按区域存，不能整帧存。
4. **小部件会被平均掉。** 整帧均匀切块时，小部件占不到一个小块；合并相似 token 时，它又最先被抹掉。所以 token 要由 affordance map 决定取在哪里。

**affordance map 在隐式记忆里的角色。** 第 3、4 点指向同一个设计：**affordance map 决定记忆把容量花在哪里。** 画面里能操作的地方存高分辨率的特征，其余只留很粗的背景。别人用"相似就合并"（MA-LMM）或"视野重叠"（WorldMem）来挑，我们用"能不能操作"来挑。我认为这是这条路线有自己特点的地方。它目前只是假设，需要试验。

**两个阶段正好对应两种读法。** 第一阶段"找到并走到跟前"只需要在特征里检索出是哪个部件、在哪一帧。第二阶段"碰到"才需要从证据帧读出几何量。

**如果采用 L1 或 L2，各环节的变化：**

| 环节 | 变化 |
| --- | --- |
| 4 affordance map | 除了分数图，还要输出可以存下来的特征；它同时决定 token 取在哪里 |
| 6 跨帧关联与合并 | L1 不需要，读取时去重；L2 用特征相似度加粗略位置，做法参考 SAM 2 和 Dynam3D，但旧状态不删 |
| 7、8 接触点、法向、站立点 | 从"写入时算好"改为"读取时现算"，算法不变 |
| 9 条目内容 | 变成部件 token 加薄索引；原来的字段清单变成读出清单 |
| 10 查询与检索 | 特征相似度 × 时间 × 位置，再回到证据帧 |
| 15 事件与状态时间线 | 事件 token（接触前后的特征），加上显式的时间和次数 |
| 13 评测 | 增加存储大小和查询耗时；其余指标照旧 |

**需要你定的点（存储形式）：**

1. 你说的 implicit 更接近哪一档：L1、L2 还是 L3；或者是前面表里的 ③（纯状态）或 ④（特征场）。
2. 时间、位姿和事件的离散部分保持显式，是否接受。
3. 接触点、法向、站立点不再存，改为读取时现算，评测照旧，是否接受。
4. 谁来读记忆：相似度检索、小的读出头、视觉语言模型看证据帧，先做哪一种。
5. 特征用哪一种：图文对齐的特征（便于文字查询）；自监督特征（便于认出同一个部件，Robo-ABC 和 D3Fields 用的是这一类）；还是两种都存。

**你自己读文章时，建议的顺序：**

| 顺序 | 文章 | 读它是为了看什么 | 从哪里读起 |
| --- | --- | --- | --- |
| 1 | WorldMem | 带位姿和时间戳的记忆单元，按状态挑记忆 | 第 3.3 节 |
| 2 | SAM 2 或 XMem | 记忆库怎么写、怎么读、怎么整理 | SAM 2 第 4 节；XMem 第 3 节 |
| 3 | Video-3D LLM | 用深度和位姿给特征加位置 | 第 3.2 节，表 3 和表 5 |
| 4 | Point3R | 一张图对比四种记忆形式 | 图 1 |
| 5 | D3Fields | 不存三维，只存每个视角的图，查询时投影 | 第 3.2 节 |
| 6 | Ego-Topo | 第一视角、不重建、按活动组织空间 | 第 3.2 节 |
| 7 | RoboMME | 同一个模型上比较文字、特征、状态三种记忆 | 表 3 |

**五点的决定（2026-11-11，第四轮讨论）：**

| # | 问题 | 你的决定 |
| --- | --- | --- |
| 1 | 隐式到哪一档 | L2：图 + token |
| 2 | 时间、位姿和事件的离散部分保持显式 | 接受 |
| 3 | 接触点、法向、站立点不再存，读取时现算 | 接受 |
| 4 | 谁来读记忆 | 先用 affordance map 上比较有特点的内容结合传统语义匹配、搜索关键帧，再交给视觉语言模型 |
| 5 | 特征用哪一种 | 两种都存 |

这些决定已经写进最上面的"总览"和各环节的正文，并在 [pipeline_examples.ipynb](pipeline_examples.ipynb) 里各做了一个示例。

### ✏️ 存储形式 · 你的更新

（写在这一行下面）

## 1 · 关键帧选取

**状态：** 真实

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 1"：相机走过的路线、关键帧的位置，以及取帧数量对部件覆盖的影响。

**想法：** 记忆以关键帧为单位，不做全局重建。从视频里挑少量帧，后面所有环节只在这些帧上计算。在 L2 记忆里，关键帧本身就是记忆的观察层。

**输入 → 输出：** 整段视频（RGB，10 FPS，1025 帧）→ 一组关键帧。

**要求（草案）：**
- 不能借助标注来选帧，否则等于提前知道部件在哪里。
- 每个功能部件至少在一帧里足够大、足够清楚。
- 帧数尽量少：关键帧的图是 L2 记忆里最占空间的部分。
- 同一方法能用于 S2 的交互视频和 S3 的长视频。

**现在：** 按时间均匀取 40 帧，工作分辨率 720×960（原始分辨率的一半）。为了能打分，只在有真值深度的帧里取。代码：`report/mem_demo_helpers.py` 的 `pick_keyframes`。

**示例里看到的：**
- 取 10、20、40、80 帧时，至少有 10 个像素可见的部件分别是 12、24、29、34 个（共 38 个）。从 40 帧加到 80 帧，多看到 5 个部件。所以 40 帧时没进记忆的 9 个部件里，至少有 5 个是取帧太少造成的。
- 均匀取帧会取到相机正在快速转动的帧。有区域的 22 个关键帧里，相机转速的中位数是 24°/s（最慢 3°/s，最快 55°/s）。转得快的帧画面模糊，而且投影的标注和画面对不上（见环节 4）。

**缺什么：**
- 没有按相机移动量或画面内容选帧，也没有剔除转得快、画面模糊的帧。
- 有人操作的视频里，要在每次交互的前后各取一帧：接触前的一帧里部件没有被手挡住，接触后的一帧记录结果。这依赖环节 14。
- 帧数再往上加（160、全部）还没试，所以还不能断定剩下的部件是不是视频里本来就看不清。

**待定：**
- 取多少帧。
- 选帧方式：按时间均匀、按相机移动量，还是按"出现了新内容"。
- 小部件处是否需要回到原始分辨率。

**建议下一步：** 在每个时间窗里选设备位姿转得最慢的一帧（只用输入），重跑示例，看标注错位、接触点误差和视觉语言模型的选择各改善多少。要给新的关键帧补算单目深度，只用 CPU。

### ✏️ 1 · 你的更新

（写在这一行下面）

## 2 · 深度

**状态：** 真实

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 2"：同一帧的 RGB、LiDAR 深度、单目深度和真值深度并排。

**想法：** 用"简单深度"代替重建：每个关键帧一张深度图，把像素抬到三维。主用单目深度，因为任何视频都能用；设备 LiDAR 只用来校正尺度，并作为对照。

**输入 → 输出：** 关键帧 RGB，加上低分辨率 LiDAR 深度（256×192）→ 每帧一张以米为单位的深度图。

**要求（草案）：**
- 尺度正确，单位是米。
- 在小部件及其周围足够准。部件只有几厘米宽，深度差几厘米，接触点就落到部件外面。
- 不依赖激光扫描。
- 没有 LiDAR 的视频也要有办法定尺度。S3 的 HD-EPIC 由 Aria 眼镜拍摄，没有深度传感器。

**现在：** Depth Anything V2（室内米制版，Large）。它的原始输出偏远约 33%，所以每帧乘一个尺度（LiDAR 与预测之比的中位数）。校正后全图中位误差 4.7 cm，部件处 11.2 cm。纯 LiDAR 是 2.4 cm 和 5.2 cm，但只覆盖 88% 的像素。代码：`affordance3d/mem/depth_models.py`，`pipeline.py` 的 `lidar_scaled`。

**缺什么：**
- 部件处误差是全图的两倍多，原因没有拆开：是细小结构被抹平，还是一个全局尺度不够。
- 只校正了一个尺度，没有校正偏移，也没有分区域校正。
- 没有比较其他深度模型。Depth Pro 在 Mac 上输出不可信，需要在服务器上重测；MoGe-2、UniDepth V2 没试。
- 没有 LiDAR 时怎么定尺度，没试。

**待定：** 小部件处用哪种办法。
- A：部件附近优先用 LiDAR，空洞处用单目深度补。
- B：用 LiDAR 做"尺度 + 偏移"或分区域校正。
- C：不追求部件本身的深度，由周围表面推算（见环节 7）。
- D：以后把接触点深度作为模型的预测目标。

**建议先试：** 在部件周围比较 A、B、C 的误差，和环节 7 一起做。只用 CPU。

### ✏️ 2 · 你的更新

（写在这一行下面）

## 3 · 相机位姿

**状态：** 真实

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 3"：设备位姿和真值位姿的对照，以及每个关键帧的误差。"环节 6"里有它对关联的影响。

**想法：** 用设备自带的位姿（ARKit）把各帧放进同一个坐标系，不做 SfM 或 SLAM。

**输入 → 输出：** ARKit 轨迹（60 FPS）→ 每个关键帧的相机位姿和重力方向。

**要求（草案）：**
- 时间相近的关键帧之间要可靠；隔得远的只要求方位和大致距离（第三轮讨论）。
- 同一个部件从相近的帧抬到三维后要落在一起（厘米级），否则环节 6 无法判断是不是同一个部件。
- 高度方向不能漂，否则地面和站立点会错。
- 只用设备数据。

**现在：** 直接使用。与真值相比，相机中心误差中位 7.2 cm，个别帧 89 cm；朝向误差中位 1.2°。高度漂移使人形双脚离真实地面 2–6 cm。为了打分，输入相机用真值配准变换到扫描坐标系，这是整体的刚体变换，不改变帧与帧之间的相对关系。代码：`affordance3d/mem/inputs.py`。

**示例里看到的：** 同一个部件在两个关键帧里各算一次位置，两次相差中位 26 cm（深度误差和位姿误差都在里面）。这和相邻部件的间距是同一个量级，所以环节 6 只靠位置分不清挨着的部件。

**缺什么：**
- 没有任何校正，也没有剔除明显有问题的帧。
- 上面的位置差里，深度和位姿各占多少没有拆开。
- 没有分析误差在时间上的分布：是否集中在某一段。

**待定：**
- 接受现状，还是在关键帧之间做轻量校正（例如用深度对齐相邻帧）。
- 是否剔除位姿明显有问题的帧。

**建议下一步：** 把位置差拆成深度和位姿两部分（各换成真值算一次），再决定先改哪一个。只用 CPU。

### ✏️ 3 · 你的更新

（写在这一行下面）

## 4 · affordance map

**状态：** ORACLE

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 4"：几个关键帧上的 affordance map（颜色是动作类型）和从图上找出的区域。

**想法：** 在每个关键帧上标出哪里可以操作、属于哪一类动作。它和深度一起是记忆的全部原料。在 L2 记忆里它还有第二个作用：**决定 token 取在哪里**，也就是记忆把容量花在画面的哪些地方。

**输入 → 输出：** 关键帧 RGB（可选深度）→ 每个像素的动作类型（9 类，或"不可操作"）。连在一起的同类像素是一个区域，每个区域之后变成一个 token。

**要求（草案）：**
- 能找到小部件：把手、旋钮、按键、插座。
- 位置准到能确定接触点。
- 动作类型正确。
- 只看这一帧或邻近几帧，不看扫描。
- 在没见过的场景上有效。SceneFun3D 有官方的训练和验证划分。
- 能与公开方法对照。
- 相邻的同类部件能分开（示例里发现的新要求）。

**现在：** ORACLE。把扫描上标注的部件点用真值相机投影到帧上（深度容差 5 cm），取标注的动作类型，**去掉部件编号**。投影得到的是稀疏的点，所以用 7 像素的闭运算把同一类动作的点填成块，让它像模型预测的稠密图。代码：`affordance3d/mem/labels.py`，`tokens.py` 的 `action_map` 和 `find_regions`。

**示例里看到的：**
- 40 个关键帧里找出 62 个区域（至少 10 个像素），覆盖 28 个真实部件。有 3 个区域把相邻的同类部件连在了一起。不做填充时，同一个部件在一帧里会碎成几块，环节 6 就连不对。
- **标注和画面并不总是对得上。** 相机转得快的关键帧里，投影的标注偏离真实部件十几个像素；转得慢的帧基本重合（"环节 4"的第二张图）。这种错位有没有反映到接触点误差上，这次看不出来：v1 的 29 个条目里，接触点误差和证据帧相机转速的秩相关系数是 −0.38，转得快的帧误差并不更大。所以它对接触点的影响还不能下结论；它对"按标注位置取特征、画框"的影响是直接看得见的。

**缺什么：**
- 标注投影的对齐。原因还没有查清：可能是画面的曝光时刻和位姿的时刻不一致，也可能是画面模糊时真值位姿本身不准。制作训练标签之前必须解决，否则标签本身是偏的。
- 模型本身。
- 批量制作训练标签的流程：现在只对一个场景做了投影。
- 可以存下来的特征：现在 token 的特征来自两个通用的现成模型，不是 affordance 模型自己的特征。
- 区分相邻同类部件的办法。
- 这一环节自己的评测指标。

**待定：**
- 模型路线：在预训练视觉特征上训练小预测头，还是先看现成的分割或视觉语言模型不训练能做到什么程度。
- 只输出每个像素的动作类型，还是同时输出部件之间的分界。
- 训练规模和算力。训练需要 GPU，要走队列。

**建议下一步：** 不训练，先看现成模型在这 40 帧上能找到多少部件，作为基线；同时把标签制作做成可以批量运行的。

### ✏️ 4 · 你的更新

（写在这一行下面）

## 5 · 部件运动

**状态：** ORACLE

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 5"：标注的运动轴画在关键帧上。运动现在只用于人形画面里的箭头，还没有进入 token。

**想法：** 记住部件怎么动：转动还是平移，绕哪根轴，朝哪个方向。它是"动作结果"的几何部分，也决定人形手的运动方向。

**输入 → 输出：** 部件在帧上的位置和深度 → 运动类型、轴的方向、轴的原点。

**要求（草案）：**
- 类型正确（转动或平移）。
- 方向准到画面上看起来对：抽屉向外，门绕铰链。
- 从输入得到。

**现在：** ORACLE。取自 SceneFun3D 的运动标注，38 个部件里 37 个有。演示里显示为黄色箭头。

**缺什么：**
- 预测方法。
- 误差度量还没用起来（轴夹角的函数已有：`actionable.py` 的 `axis_angle_deg`）。
- 数据集里没有运动范围（能开多大）。
- S1 是静态视频，看不到部件真的在动，只能从外观推断；S2 的交互视频里才有真实运动。

**待定：**
- S1 先保留标注，到 S2 再从交互视频里估计，还是 S1 就从外观预测。前一种正好是"交互帮助构建记忆"的检验点。
- 表示方式：直接回归方向，还是利用表面法向和重力来约束。

**建议先试：** 检查标注的轴是否几乎总是平行于重力、表面法向，或两者的叉积。如果是，运动可以由"类型 + 表面几何"得到，不必回归任意方向。只用 CPU。

### ✏️ 5 · 你的更新

（写在这一行下面）

## 6 · 跨帧关联

**状态：** 真实（初版）

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 6"：连成同一个部件的 token 按时间排开，以及两张说明"为什么难"的图。

**想法：** 同一个部件会出现在多个关键帧里。把这些 token 连到同一个部件上，查询时才能把多帧的证据合起来，也才能说出"这个部件什么时候被看到过"。连起来的 token 全部保留，不取平均：取平均会抹掉小部件的细节。

**输入 → 输出：** 所有 token（特征和薄索引），加上各关键帧的深度和位姿 → 每个 token 属于哪个部件。

**要求（草案）：**
- 同一个部件不被拆成几个；相邻的不同部件（一排旋钮、一排抽屉把手）不被连在一起。
- 不使用标注的部件编号。
- 能随时间增量进行：新的关键帧来了，只和已有的部件比较。S2 和 S3 需要。
- 旧的 token 不删。场景变了以后，旧状态要留着回答"之前是什么样"。

**现在：** 按时间顺序逐个处理 token。在动作类型相同、位置相距不超过 30 cm、视觉特征相似度不低于 0.5 的已有部件里，选位置最近的一个连上；没有就新建一个部件。同一个关键帧里的两个区域不会连到同一个部件。位置由 token 自己那一帧的像素、深度和位姿现算。代码：`tokens.py` 的 `link_tokens`。

**示例里看到的：**
- 62 个 token 连成了 39 个部件，真实部件是 28 个；完全连对的有 14 个。
- 94% 的 token 所在的部件没有混进别的部件；一个真实部件平均有 83% 的 token 留在同一个部件里。
- 同一个部件两次看到时，算出的位置相差中位 26 cm，和相邻部件的间距是同一个量级。
- 挨得近的同类部件视觉特征几乎一样，相似度帮不上忙，只能靠位置。

**缺什么：**
- 更可靠的位置依据。现在每个 token 的位置各算各的，深度误差和位姿误差都在里面。
- 同一帧里部件之间的相对位置还没用上：相邻的两个旋钮在任何一帧里都是"左边那个"和"右边那个"。
- 置信度：见过几次，各次是否一致。

**待定：**
- 关联在写入时做（现在的做法），还是只在检索之后、在候选关键帧之间做。后一种更符合"需要时再算"，也不必一次连好整段视频。
- 是否用成对匹配（两个关键帧之间一次配好所有区域）代替逐个贪心。
- 阈值怎么定。30 cm 是按位置误差定的，换设备和场景可能不同。

**建议下一步：** 用同一帧里的相对位置做成对匹配，看"完全连对"能提高多少。只用 CPU。

### ✏️ 6 · 你的更新

（写在这一行下面）

## 7 · 接触点

**状态：** 真实，读取时现算

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 7、8"：检索选出的 token 在证据帧上的接触点、法向和站立点。

**想法：** 手或脚应当碰到的那个三维点，是"位置准确"的核心量。它不存进记忆：查询选出 token 之后，回到这个 token 的证据帧，用那一帧的深度现算。这是精度的第二个阶段（"碰到"）；第一个阶段（"找到并走到跟前"）由检索负责。

**输入 → 输出：** token 的区域（证据帧上的像素）、这一帧的深度和位姿 → 一个三维点。

**要求（草案）：**
- 落在部件上。目前的标准是离真值部件不超过 3 cm。
- 对深度噪声稳健。
- 在人够得着的那一面，不在部件背面或内部。
- 算得快：每个查询都要算一次。

**现在：** 取区域的中心像素，深度取区域内深度的中位数，只用证据帧。代码：`pipeline.py` 的 `contact_and_normal`，`tokens.py` 的 `read_geometry`。

**示例里看到的：**

| 记忆 | 部件数 | 中位误差 | 落在部件上 |
| --- | --- | --- | --- |
| L2：读取时现算，不知道部件编号 | 39 | 11.1 cm | 13% |
| v1：写入时按部件编号算好存下 | 29 | 7.7 cm | 14% |

两行用的是同一帧的同一种深度（单目深度 × LiDAR 尺度）。两行的部件数不同：L2 把 28 个真实部件记成了 39 个，多出来的是没有连上的重复观察。这可能是它的中位误差更大的原因，还没有单独验证。 现算一次约 17 毫秒。v1 的另外两组对照（LiDAR：2.8 cm、54%；真值深度和位姿：0.4 cm、97%）说明计算链路本身是对的，误差来自深度和位姿。

**缺什么：**
- 小部件处的深度（环节 2）。
- 同一个部件的多个 token 还没有一起用：现在只用像素最多的那一个。
- 没有利用周围的表面。
- 接触点是区域中心，没有考虑该抓哪里，例如门把手应当握在远离转轴的一端。

**待定：**
- 提高精度的办法：A 用同一部件的多个 token 取中位数；B 沿视线与周围支撑面求交，再加一个小偏移；C 优先用 LiDAR；D 到了跟前重新看一眼。
- 接触点的定义：区域中心，还是随动作类型而定。

**建议下一步：** A、B、C 单独和组合的误差表。只用 CPU。

### ✏️ 7 · 你的更新

（写在这一行下面）

## 8 · 法向、地面与站立点

**状态：** 真实，读取时现算

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 7、8"（与接触点画在同一张图上）。

**想法：** 人形从哪里、朝哪个方向接近部件。法向给出部件朝外的方向，地面给出脚的高度，站立点是人站的位置。和接触点一样，读取时从证据帧现算。

**输入 → 输出：** 接触点、证据帧的深度、证据帧前后几个关键帧观察到的点 → 支撑面法向、站立位置和朝向、是否空闲。地面高度是写入时存下的唯一一个场景级的数。

**要求（草案）：**
- 站立点在地面上，没有被占据，而且确实被看到过是空的。
- 从站立点手够得到接触点，必要时前倾或蹲下。
- 朝向面对部件。
- 部件在台面上方或朝上时，也有合理的站位。

**现在：**
- 法向：接触点周围 48×48 像素内（去掉区域本身的像素）拟合平面。
- 地面：写入时由所有关键帧观察到的点得到一个高度：最低 20% 的点里出现最多的那一档（每档 2 cm）。
- 站立点：沿水平法向后退 0.5 m（用脚的动作 0.45 m）；被挡就再退 0.15、0.3、0.45 m，还不行就转 ±30°、±60°。法向接近竖直时，改为从拍摄方向接近。
- 是否空闲：身体当作半径 0.2 m 的圆柱，只与证据帧和它前后各 3 个关键帧观察到的点（5 cm 体素）比较，不融合整段视频。
- 代码：`affordance3d/mem/actionable.py`，`tokens.py` 的 `read_geometry`。

**示例里看到的：** 站立点在真值扫描里空闲的比例，L2 是 74%，v1（与全部 40 个关键帧的点比较）是 93%。L2 低了 19 个百分点。只看相邻几帧时，没看到的空间更多，而没看到的空间被当成空闲，所以选出的站立点更容易其实被占着。

**缺什么：**
- 没看到的空间被当成空闲，与"未知不等于空闲"的原则不符。只看相邻几帧以后，没看到的空间更多。
- 不检查够不够得着：退得越远越够不着。
- 拟合窗口按像素而不是按米，远处部件的窗口实际很大。
- 法向没有稳健拟合，窗口跨两个面时会歪；也没有量过法向误差（真值可以从扫描算）。
- 地面只有一个高度，有台阶的场景不适用。

**待定：**
- 空闲检查看多少个相邻关键帧；或者改为只接受"确实看到是空的"的位置。
- 够得着的检查用什么形式。
- 多个候选站立点如何打分（距离、朝向、空闲程度）。

**建议下一步：** 把空间分成空闲、占据、未知三种，加上够得着的约束，重新统计"确实空闲且够得着"的比例。只用 CPU。

### ✏️ 8 · 你的更新

（写在这一行下面）

## 9 · 部件 token（原"条目内容"）

**状态：** 真实（现成特征，未训练）

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 9"和"记忆里到底存了什么、有多大"。

**想法：** 记忆里每个部件存什么。第四轮讨论定为 L2：不存一条条写好的字段，存 **token**。一个 token 对应某个关键帧里的一个可操作区域，内容是特征（隐式），外加一层很薄的索引（显式）。"它是什么、能怎么用、现在什么样"在特征里；"什么时候、在哪一帧的哪里"在索引里。

**一个 token 里有什么：**

| 部分 | 内容 | 用来做什么 |
| --- | --- | --- |
| 视觉特征（隐式） | 自监督模型（DINOv2）在区域上的平均特征，384 维 | 认出同一个部件（环节 6） |
| 语义特征（隐式） | 图文模型（SigLIP 2）对"区域连同周围一圈"的编码，768 维 | 和文字查询比较（环节 10） |
| 薄索引（显式） | 时间、关键帧编号、区域中心的像素、区域的深度、像素数、动作类型 | 按时间和位置筛；回到证据帧 |

**不在 token 里的：** 接触点、法向、站立点（读取时现算，环节 7、8）；任务句子和部件编号（标注，只用于打分）；部件运动（环节 5，还没有进来）。

**要求（草案）：**
- 能回答五个问题：在哪里；能做什么；做了会怎样；怎么做；凭什么（证据帧）。前四个由读取时的读出回答，第五个由索引回答。
- 支持环节 10 的各类查询。
- 每个 token 都能指回它的证据帧，可以逐条检查。
- 容量花在可操作的区域上，不按整帧均匀存。
- 为事件留位置（环节 15）。

**现在：** 每个区域一个 token，两种特征都存。代码：`affordance3d/mem/tokens.py`（`PartToken`、`L2Memory`、`write_memory`），`encoders.py`。

**示例里看到的：**
- 写入一次用了 47 到 89 秒（CPU）：40 个关键帧得到 62 个 token。
- 实测大小：关键帧原图 13.64 MB，深度图 12.72 MB，affordance map 0.10 MB，全部 token 的特征 143 KB，薄索引 5.3 KB。对照：v1 的显式条目 15.6 KB，整个房间的激光扫描 133.21 MB。
- token 很小，记忆的大小几乎全在关键帧的图上。

**缺什么：**
- **动作结果。** token 里没有"做了会怎样"。在 L2 里它应当是事件 token 的内容（接触前后的特征），S1 的视频里没有。
- 特征没有针对 affordance 训练：两个都是通用的现成模型。
- 所属物体的语义：现在只靠裁剪图周围的一圈画面带进来。
- 置信度。
- 动作类型现在是索引里的一个显式标签，来自 ORACLE 的 affordance map。

**待定：**
- 语义特征裁多大一圈：太小看不出它属于什么物体，太大又把别的东西带进来。
- 每个部件留多少个 token：全部保留，还是只留最清楚的几个。
- 关键帧的图是否降分辨率，或只留可操作区域附近，来减小记忆。
- 动作类型保持显式标签，还是也变成特征。

**建议下一步：** 换几种裁剪大小，看环节 10 的检索结果怎么变。只用 CPU。

### ✏️ 9 · 你的更新

（写在这一行下面）

## 10 · 查询与检索

**状态：** 真实（现成模型，未训练）

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 10a"和"环节 10b"。

**想法：** 用一句话从记忆里找到要操作的部件。第四轮讨论定为两步：**先用 affordance map 上有特点的内容结合传统语义匹配、搜索关键帧，再把找到的关键帧交给视觉语言模型。** 第一步快而粗，负责"在哪几帧里"；第二步慢而细，负责"是哪一个"。

**输入 → 输出：** 文字查询 → 一个 token，以及它所属的部件和证据帧。之后由环节 7、8 现算几何量。

**要求（草案）：**
- 支持按结果查（"打开落地灯"）、按动作查（"哪些东西可以拉开"）、按位置或物体查；有了事件之后，支持按时间和发生过的动作查。
- 返回的结果带证据帧。
- 有公平的对照：只用语义、不用 affordance 的检索（讨论稿第 4 节的对照组）。
- 有真值可评：SceneFun3D 标注了每条任务句子对应哪个部件。
- 第一步要快：记忆变长以后，它决定查询的耗时。

**现在：**
- **第一步，找关键帧。** 两种线索相加。传统语义：查询文字和整个关键帧的图文相似度。affordance token：查询文字和每个 token 的语义特征的相似度，一个关键帧取它最好的 token。两者各自标准化后相加，取前 3 个关键帧。
- **第二步，视觉语言模型。** 把这 3 个关键帧上的每个区域画框、编号，问模型"要完成这个任务应该操作几号"。模型是 Qwen3-VL-4B，在 CPU 上运行，没有训练。
- 代码：`affordance3d/mem/search.py`，`encoders.py`。

**示例里看到的**（24 条任务句子里，有 18 条所指的部件出现在关键帧里）：

| 找关键帧的办法 | 第 1 个就对 | 前 3 个里有对的 |
| --- | --- | --- |
| 只用传统语义（对照） | 67% | 83% |
| 只用 affordance token | 33% | 72% |
| 两者相加（现在的做法） | 83% | 100% |
| 两者相加，再加上从文字猜的动作类型 | 83% | 100% |

- 找关键帧每个查询约 0.2 毫秒。
- 视觉语言模型在 18 个任务里选对了 10 个部件，每个查询约 60 秒（CPU）。这 18 个任务的前 3 个关键帧里都有正确的部件，所以错的 8 个都错在第二步。其中 4 个是阳台门的任务，模型都答了同一个编号。
- 从任务文字猜动作类型只对了 18 个里的 7 个。"打开"可以是拉、转或者踩，光看文字分不出来，所以现在没有用它排序。

**缺什么：**
- 动作类型的用法。它是 affordance map 上最有特点的信息，但从文字里猜不准。
- 小部件在整幅关键帧里很小，视觉语言模型不容易看清。
- 时间和动作线索：要等事件层（环节 15）。
- 更多场景上的评测；现在只有 18 个任务。

**待定：**
- 动作类型怎么进入第一步：先由语言模型把查询改写成"要找的部件 + 要做的动作"；还是第一步不用，留给视觉语言模型。
- 交给视觉语言模型的形式：整幅关键帧加编号框（现在的做法），还是每个候选区域的放大图。
- 取前几个关键帧。
- 视觉语言模型用哪一个，是否上 GPU（要走队列）。

**建议下一步：** 把交给视觉语言模型的图换成候选区域的放大图，比较选对的比例。只用 CPU。

### ✏️ 10 · 你的更新

（写在这一行下面）

## 11 · 末端姿态

**状态：** 真实

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 11、12"：用读取时现算的接触点、法向和站立点摆放的人形。

**想法：** 人形最后的姿势：站在站立点，面对部件，手或脚到接触点。画面要让人一眼看出位置准、动作对。

**输入 → 输出：** 读取时现算的接触点、法向和站立点，token 的动作类型，和地面高度 → SMPL-X 姿态。

**要求（草案）：**
- 手或脚落在部件上。
- 姿态自然：不穿过家具，关节不反折。
- 不同的动作类型看得出区别。
- 脚在地面上。

**现在：** 手掌瞄准接触点前方 3 cm。接触点低于 0.55 m 时蹲下。部件偏左用左手，否则用右手；foot_push 用右脚。姿态由 300 步迭代求出，只调脊柱和一条手臂（或一条腿）。没有手指动作。演示的 4 个查询里，手或脚到真值部件 1.6–8.6 cm，脚离真实地面 2.3–5.5 cm，身体与扫描的冲突不超过 0.1%。代码：`affordance3d/mem/humanoid.py`。

**缺什么：**
- 9 类动作只有伸手和伸脚两种姿态，捏、勾、按、转没有区别。
- 手的朝向没有约束，手掌不一定对着表面。
- 不演示动作本身：沿运动轴拉、转。
- 隔着台面够不到时，不会前倾或侧身。
- 穿模只是事后统计，没有作为约束。
- 只有一种体型。

**待定：**
- 每类动作一个手型预设是否足够。
- 是否加动作演示：手沿运动轴移动一小段，部件本身不动。
- 现在做，还是等生成式动作（环节 12）一起做。

**建议先试：** 给 9 类动作各做一个手型，在同一个部件上并排画出来比较。只用 CPU。

### ✏️ 11 · 你的更新

（写在这一行下面）

## 12 · 走近过程与画面

**状态：** 真实

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 11、12"：贴在视频帧上的人形、扫描里的检查视角，以及一段走近的短视频。

**想法：** 人形怎么走到部件前，以及最后给人看的画面。

**输入 → 输出：** 站立点、末端姿态、视频帧 → 一段动画和静态图。

**要求（草案）：**
- 走的路径不穿过家具。
- 步态自然，脚不滑。
- 画面里同时看得到人和部件，遮挡关系正确。
- 能批量生成，每个查询一段。

**现在：** 起点在站立点朝相机方向最多 1.5 m 处。直线走 18 帧，再用 12 帧伸手。步态是简单的周期摆动。展示帧从整段视频里挑选，标准是部件可见、人在画面内、距离合适。人体用输入深度做遮挡。另有一个在扫描里的检查视角。代码：`humanoid.py` 的 `approach_frames`，`report/mem_demo_helpers.py`。

**缺什么：**
- 路径规划：现在不避障。
- 真实的步态：现在脚会滑。
- 从走到伸手的过渡是线性插值。
- 背景是静止的一帧，相机不动。
- 生成式动作模型（LINGO、DartControl）还没有安装。

**待定：**
- 是否换成生成式动作。需要 GPU，要走队列。
- 画面形式：贴在真实视频帧上、在三维里自由视角，还是两者并排。
- 需要细致到什么程度。

**建议先试：** 先定画面形式，再决定是否上生成式动作。

### ✏️ 12 · 你的更新

（写在这一行下面）

## 13 · 评测

**状态：** 真实，单场景

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 13"和"记忆里到底存了什么、有多大"。

**想法：** 用激光扫描和标注给每个环节打分，并回答两个核心问题：加入动作信息的记忆是否更好；不做稠密重建能省多少、会差多少。

**要求（草案）：**
- 每个环节有自己的指标，能看出误差从哪里来。
- 多个场景，有训练和测试的划分。
- 与公开方法可比。
- 有对照组：只用语义的检索（讨论稿第 4 节）；先重建再读取的记忆（第三轮讨论）。
- 报告成本：存储大小、写入耗时、每个查询的耗时。
- 精度分两个阶段报告：先"找到并走到跟前"，再"碰到"。

**现在：** 一个场景，一个视频。
- 几何：接触点误差、落在部件上的比例（不超过 3 cm）、站立点在扫描里是否空闲、手或脚到部件的距离、脚离地高度、身体与扫描的冲突。
- 关联：完全连对的部件数、token 所在部件有没有混进别的部件。
- 检索：前 1 个、前 3 个关键帧里有没有正确部件；视觉语言模型选对的比例。
- 成本：各部分的字节数；写入和读取的耗时。
- 代码：`report/mem_demo_helpers.py` 的 `scoreboard`、`shots_table`，`report/mem_steps_helpers.py`。

**示例里看到的成本：**

| 项目 | 大小或耗时 |
| --- | --- |
| 关键帧原图（40 帧） | 13.64 MB |
| 深度图 | 12.72 MB |
| affordance map | 0.10 MB |
| 全部部件 token 的特征 | 143 KB |
| v1 的显式条目（对照） | 15.6 KB |
| 激光扫描（真值，不是输入） | 133.21 MB |
| 写入记忆 | 47 到 89 秒，每段视频一次 |
| 找关键帧 | 0.2 毫秒，每个查询 |
| 视觉语言模型读 3 个关键帧 | 60 秒，每个查询（CPU） |
| 现算接触点、法向、站立点 | 17 毫秒，每个查询 |

**缺什么：**
- 多场景。
- **重建对照组**：同一段视频先融合点云再读接触点，与 L2 比较精度、耗时和存储。按第三轮讨论，这是 S1 的下一步。
- 换成预测后需要的指标：部件发现率、动作类型准确率、运动轴误差。
- SceneFun3D 官方指标。它们定义在激光扫描的点上，我们需要把预测投回扫描才能比；而且公开方法以扫描为输入，设定不同。
- 时间类问题的评测，要等事件层。

**待定：**
- 开发阶段用多少个场景。
- 哪些指标作为主指标。

**这个环节随每个环节一起补，不单独做。**

### ✏️ 13 · 你的更新

（写在这一行下面）

## 14 · 交互检测（新增，提案）

**状态：** 缺（S1 的视频里做不了）

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 14、15"：只有一张示意图，没有数据。

**想法：** 在第一视角视频里找出实际发生的动作：什么时候、哪只手、碰了哪个部件、做了哪一类动作。它是事件的来源，也是 affordance 的证据：被操作过的地方一定可以操作。

**输入 → 输出：** 视频帧，加上 affordance map → 交互片段：起止时间、哪只手、接触位置、被操作的部件、动作类型。

**要求（草案）：**
- 起止时间准到能取出"接触前"和"接触后"的帧。
- 能对应到部件层的条目上。
- 动作类型用同一套 9 类。
- 不依赖重建。

**现在：** 没有。S1 的视频里没有人操作。

**缺什么：**
- 整个环节。
- 数据：S2 的 FunGraph3D 交互视频没有时间标注；S3 的 HD-EPIC 有动作区间、手部掩码和物体搬移轨迹。

**待定：**
- 先用数据集标注的动作区间（ORACLE），还是直接用现成的检测器（100DOH、Hands23）。
- 接触位置怎么定：EPIC-Aff 取手和物体两个框重叠部分的中心。
- 动作类型怎么得到：由被操作部件的 affordance 类型直接给出，还是另外识别。

**建议先试：** 拿一段带交互的视频跑现成的手—物检测器，看起止时间和接触位置的质量。

### ✏️ 14 · 你的更新

（写在这一行下面）

## 15 · 事件与状态时间线

**状态：** 缺（S1 的视频里做不了）

**示例：** [pipeline_examples.ipynb](pipeline_examples.ipynb) 的"环节 14、15"：一张示意图，没有数据。

**想法：** 把"做过什么"写进记忆。按第四轮讨论定下的形式，每个事件是一个**事件 token**：内容是接触前和接触后的特征（隐式），外加显式的离散部分。部件的"状态"不单独存，它就是这个部件最近一次事件之后的样子。

**一个事件 token 里有什么（设计，还没有实现）：**

| 部分 | 内容 |
| --- | --- |
| 特征（隐式） | 接触前一帧里部件区域的特征，接触后一帧里的特征；两种特征都存，与部件 token 相同 |
| 离散部分（显式） | 起止时间；挂在哪个部件上；是这个部件的第几次；动作类型 |
| 证据 | 接触前、接触后两个关键帧的编号 |

**输入 → 输出：** 交互片段（环节 14）和部件 token（环节 9）→ 事件 token，按时间排在各自的部件上。

**要求（草案）：**
- 每个事件挂在一个部件上，并带接触前后的证据帧。
- 没有观察到的时段记为未知，不把旧状态当成事实。
- 能回答"上次是什么时候"、"现在是什么状态"、"在某个动作之前或之后"。时间和次数靠显式的记录；状态本身靠视觉语言模型看接触后的证据帧。
- 可以关掉事件的特征，退化成只有时间戳的记忆，用作对照。

**现在：** 没有。部件 token 里只有"在哪些关键帧见过"的时间。

**缺什么：**
- 实现。
- 带交互的数据。按第三轮讨论，S1 的重建对照之后，先用一段 HD-EPIC 视频做最小版本。
- 真值：HD-EPIC 没有部件状态的标注。

**待定：**
- 状态是否完全不用词表（只靠特征和视觉语言模型现读），还是给常见的状态（开或关）留一个显式标签。文献里计数和离散的进度用显式记录更可靠，见"隐式还是显式"一格。
- 状态有没有变，由视觉语言模型比较前后两帧来判断，还是比较前后两个特征的距离。
- 事件是否记录操作者当时的位置，重放时需要。

**这个环节先定设计，再到带交互的视频上做。**

### ✏️ 15 · 你的更新

（写在这一行下面）

## 依赖和建议顺序

- **环节 9 最先定。** 它决定环节 4、5 要预测什么，也决定环节 10 能查什么。
- **环节 2、6、7 一起看。** 三者共同决定接触点精度，这是目前最大的误差。
- **现在就能试（只用 CPU）：** 1、2、3、5 的轴检查、6、7、8、10、11。
- **需要 GPU，要走队列：** 4 的训练，12 的生成式动作。

建议顺序：9 → 2 和 7 → 6 → 8 → 10 → 4 和 5 → 11 和 12；环节 13 随每个环节一起补。顺序由你定。

**第四轮讨论之后（2026-11-11）：**

已经完成：存储形式定为 L2；每个环节在场景 421069 上有了一个示例（[pipeline_examples.ipynb](pipeline_examples.ipynb)）。上面按 v1 写的依赖和顺序保留作参考。

接下来的顺序（方案 B 不变）：

1. 你看示例，在各环节的格子里提意见，我按意见改。
2. 重建对照试验（环节 13）：同一段视频先融合点云再读接触点，和 L2 比较精度、耗时和存储。只用 CPU。第六轮（2026-11-12）你批准了：它和关键帧的修改都并入阶段 1，在 30 个验证场景上做，规格见讨论稿第 21 节。
3. 用一段 HD-EPIC 视频做事件层的最小版本（环节 14、15、10）。需要先下载这段视频并确认许可。
4. 然后是 S2 和 S3 的其余部分。

### ✏️ 总体更新

不属于某一个环节的想法、顺序上的调整、新增或合并环节，都写在这里。

（写在这一行下面）